# TCExam-XLSX &rarr; eduExaminer-EXZ mit Kommazahl-Punkten

Ausbau des Notebooks aus [mckoh/eduexaminer-exchange](https://github.com/mckoh/eduexaminer-exchange)
zu einem durchgehenden Workflow.

**Warum der Umweg:** Der CSV/XLSX-Import von eduExaminer nimmt nur ganzzahlige Punkte.
Ueber das EXZ-Format lassen sich Kommazahlen einschleusen.

**Kette:** `EXZ` &rarr; zlib &rarr; `JSON` &rarr; Base64-Felder &rarr; `XML` (Frage + Bewertung)

**Punkteregel** (`richtig = 4/k`, `falsch = -4/(n-k)`) &mdash; Erwartungswert beim Raten = 0,
max. 4 Punkte je Frage:

| korrekte Antworten k (von n=4) | richtig | falsch |
|---|---|---|
| 1 | 4 | -1,333 |
| 2 | 2 | -2 |
| 3 | 1,333 | -4 |

In [ ]:
import sys

sys.path.insert(0, ".")

from exz.excel import read_questions
from exz.exzfile import b64_decode_text, b64_encode_text, find_questions, read_exz, write_exz
from exz.qti import build_evaluation, build_presentation
from exz.scoring import compute, fmt

XLSX = "input/TCExam_datenbanken II & big data.xlsx"
SHEET = "test"
TEMPLATE = "input/export.exz"   # echtes eduExaminer-Export-File als Strukturvorlage
OUTPUT = "output/import.exz"

## 1. Excel einlesen

Die `true`/`false`-Spalten werden typtolerant gelesen: echte Booleans (`WAHR`/`FALSCH`,
wie sie ein deutsches Excel erzeugt), die Textwerte `true`/`false`, sowie `1`/`0`,
`x`, `ja`/`nein`. Geschuetzte Leerzeichen (`\xa0`) werden bereinigt.

In [ ]:
questions, warnings = read_questions(XLSX, SHEET)

for w in warnings:
    print("WARNUNG:", w)

print(f"{len(questions)} Fragen gelesen\n")
q = questions[0]
print(q.text)
for opt in q.options:
    print("  [x]" if opt.correct else "  [ ]", opt.text[:70])

## 2. Punkte vorrechnen

In [ ]:
from collections import Counter

verteilung = Counter()
print(f"{'Nr':>3} {'n':>2} {'k':>2} {'richtig':>8} {'falsch':>8}  Frage")
for q in questions:
    p = compute(q.n_options, q.n_correct)
    verteilung[q.n_correct] += 1
    print(f"{q.number:>3} {q.n_options:>2} {q.n_correct:>2} "
          f"{fmt(p.correct):>8} {fmt(p.wrong):>8}  {q.text[:52]}")

print("\nVerteilung nach Anzahl richtiger Antworten:", dict(sorted(verteilung.items())))

## 3. XML erzeugen

`presentation` = Fragetext + Antwortoptionen (Format 1:1 aus dem Original-Notebook).
`evaluation` = die Punkte &mdash; das ist der Teil, wegen dem der ganze Umweg stattfindet.

In [ ]:
q = questions[1]
p = compute(q.n_options, q.n_correct)

print(build_presentation(q))
print("\n" + "=" * 70 + "\n")
print(build_evaluation(q, p))

## 4. Vorlage inspizieren

Zeigt, welche Felder eine Frage-Revision im echten Export hat und welches davon
die Bewertung traegt. **Das ist der Schritt, der die Feldnamen verifiziert** &mdash;
das Original-Notebook zeigt nur `presentation`.

In [ ]:
data, container = read_exz(TEMPLATE)
print("Container:", container.kind)

_, qlist = find_questions(data)
rev = qlist[0]["revisions"][0]
print("Revision-Felder:", list(rev.keys()))

for key, value in rev.items():
    xml = b64_decode_text(value)
    if xml and xml.lstrip().startswith("<"):
        print(f"\n----- {key} -----")
        print(xml[:1200])

## 5. EXZ bauen

Die erste Frage der Vorlage dient als Prototyp: sie wird je Excel-Zeile kopiert,
IDs werden neu vergeben, `presentation` und `evaluation` ersetzt. Alle uebrigen
Felder bleiben unangetastet &mdash; so bleibt die Struktur importierbar.

In [ ]:
import copy
import uuid

from exz.__main__ import _refresh_ids, _set_title

PRES_FIELD = "presentation"
SCORE_FIELD = "evaluation"   # ggf. an Schritt 4 anpassen

prototype = copy.deepcopy(qlist[0])
neu = []
for q in questions:
    p = compute(q.n_options, q.n_correct)
    item = copy.deepcopy(prototype)
    _refresh_ids(item, f"q{q.number}")
    revision = item["revisions"][0]
    item["revisions"] = [revision]
    revision[PRES_FIELD] = b64_encode_text(build_presentation(q))
    revision[SCORE_FIELD] = b64_encode_text(build_evaluation(q, p))
    _set_title(revision, q)
    _set_title(item, q)
    neu.append(item)

qlist[:] = neu
write_exz(OUTPUT, data, container)
print(f"{len(neu)} Fragen -> {OUTPUT}")

## 6. Rueckleseprobe

In [ ]:
check, _ = read_exz(OUTPUT)
_, geprueft = find_questions(check)
print(f"{len(geprueft)} Fragen wieder eingelesen\n")
print(b64_decode_text(geprueft[1]["revisions"][0][SCORE_FIELD]))